<H1>Naive Bayes Classifier for Newsgroups data using TensorFlow Probability</H1>

This notebook demonstrates how to classify newsgroups articles into various categories $C$ according to their textual contents $X$.

$\text{Let }m\text{ be the number of words in a dictionary.}$<br>
$\text{Let }n\text{ be the number of possible classes.}$<br>


$\text{Let }X_{i}\text{ be a random vector }\left[x_{0}, x_{1}, ..., x_{m}\right]\text{ representing a sequence of words (i.e. tokens) present in the article.}$<br>
$\text{Let }C_{i}\text{ be a random variable denoting the class identity of some article }x_{i}\text{ where }C_{i}=c_{k}\text{ and }k\in\left[0, 1, ..., n\right]$<br>


We can use Bayes Rule to compute the posterior probability of a newsgroup article $X=x_{i}$ belonging to category $C=c_{k}$<br> 

$$\large P(C=c_{k}|X=x)=\frac{P(X=x|C=c_{k})P(C=c_{k})}{P(X=x)}$$

The denominator $P(X=x)$ is constant since this quantity is computed from the data, therefore we can estimate the posterior $P(C=c_{k}|X=x)$ using only the numerator.

$$\large P(C=c_{k}|X=x)\propto P(X=x|C=c_{k})P(C=c_{k})$$


Naive Bayes classifies an article $X_{i}=x$ to belong to category $C_{k}$ by finding the maximum a posteriori probability  of a newsgroup article $X=x_{i}$ belonging to category $C_{i}=C_{k}$.<br>

$\text{Decision Rule: Document }X_{i}=x_{i}\text{ belongs to class }C_{i}=c_{k}\text{ which maximizes the a posteriori probability of an article text }X_{i}=x_{i}$ belonging to category $C_{i}=c_{k}$<br>

$$\large \underset{MAP}{c}=P(C_{i}=c_{k}|X_{i}=x_{i})\propto \underset{k}{\mathrm{argmax}}P(X_{i}=x_{i}|C_{i}=c_{k})P(C_{i}=c_{k})$$


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp

print("TF version:", tf.__version__)
print("TFP version:", tfp.__version__)

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import f1_score

<H1>1) Fetch and preprocess the data</H1>

In [ ]:
# Making a function get_data which:
#   1) Fetches the 20 newsgroup dataset
#   2) Performs a word count on the articles and binarizes the result
#   3) Returns the data as a numpy matrix with the labels

def get_data(categories):
    # Fetch the 20 newsgroup training data set
    newsgroups_train_data = fetch_20newsgroups(data_home='20_Newsgroup_Data/', subset='train', categories=categories)
    # Fetch the 20 newsgroup test data set
    newsgroups_test_data = fetch_20newsgroups(data_home='20_Newsgroup_Data/',  subset='test', categories=categories)
    # Save the number of documents (i.e. articles) in the training set
    n_documents = len(newsgroups_train_data['data'])
    # Create a CountVectorizer object to perform a binary word count
    count_vectorizer = CountVectorizer(input='content', binary=True, max_df=0.25, min_df=1.01/n_documents) 
    # Binarize the presence (1) or absence (0) of words in the articles in the training set
    train_binary_bag_of_words = count_vectorizer.fit_transform(newsgroups_train_data['data'])
    # Binarize the presence (1) or absence (0) of words in the articles for the test set 
    test_binary_bag_of_words = count_vectorizer.transform(newsgroups_test_data['data']) 

    return (train_binary_bag_of_words.todense(), newsgroups_train_data['target']),  (test_binary_bag_of_words.todense(), newsgroups_test_data['target']), n_documents


$\large\quad\text{Set the categories of interest and fetch/pre-process the training and test data and labels.}$<br>

In [ ]:
# The 20 categories of newgroups we have in the training data are as follows:
categories = ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

# Get the newsgroup training and test data for the categories above
(train_data, train_labels), (test_data, test_labels), num_training_documents = get_data(categories)


In [ ]:
# Defining a function to conduct laplace smoothing. This adds a base level of probability for a given feature
# to occur in every class.

def laplace_smoothing(labels, binary_data, n_classes):
    # Compute the parameter estimates (adjusted fraction of documents in class that contain word)
    n_words = binary_data.shape[1]
    alpha = 1 # parameters for Laplace smoothing
    theta = np.zeros([n_classes, n_words]) # stores parameter values - prob. word given class
    for c_k in range(n_classes): # 0, 1, ..., 19
        class_mask = (labels == c_k)
        N = np.sum(class_mask) # number of articles in class
        theta[c_k, :] = (binary_data[class_mask, :].sum(axis=0) + alpha)/(N + alpha*2)

    return theta

$\large\text{Smooth the counts to avoid divide by zero exceptions}$<br>

In [ ]:
# The binary bag of words is a 1/0 representation of whether a word occurs in a document or not
# 0's create a problem when dividing by zero so...
# Smooth the counts of the training data using Laplace smoothing to avoid division by zero
smoothed_counts = laplace_smoothing(labels=train_labels, binary_data=train_data, n_classes=len(categories))


<H1>2) Create, train and test the Naive Bayes Classifier</H1>

$\large\text{Use the smoothed counts to generate a multi-variate Bernoulli}$<br>

In [ ]:
# Create a Bernoulli distribution of having batch_shape=number of classes and event_shape=number of features.
def make_distributions(probs):
    batch_of_bernoullis = tfp.distributions.Bernoulli(probs=probs) # shape (n_classes, n_words)
    dist = tfp.distributions.Independent(batch_of_bernoullis, reinterpreted_batch_ndims=1)
    return dist

In [ ]:
# Use the smoothed counts to parameterize the multi-variate Bernoulli  
tf_dist = make_distributions(smoothed_counts)

$\large\text{Compute the prior }P(C=c_{k})$<br>
$P(C=c_{k})=\frac{\text{\# Occurences }c_{k}}{\text{\# Training Samples}}$<br>

In [ ]:
# Function which computes the prior probability of every class based on frequency its occurence in the training dataset
def class_priors(n_classes, labels):
    # Create an empty array of zeros, one for each class
    counts = np.zeros(n_classes)
    # Count and store the number of occurences of class c_k
    #class_counts = tf.math.bincount(labels, minlength=n_classes, maxlength=n_classes, dtype=tf.float32)
    # Loop over each class
    for c_k in range(n_classes):
        counts[c_k] = np.sum(np.where(labels==c_k, 1, 0))    # The prior for each class c_k is simply (# c_k occurences / total number of occurences)
    priors = counts / np.sum(counts)
    print('The class priors are {}'.format(priors))
    return priors

In [ ]:
# Compute the class priors 
priors = class_priors(n_classes=len(categories), labels=train_labels)


$\large\text{Learn the parameters that maximize(minimize) the likelihood(negative log likelihood) }P(X=x|C=c_{k})$<br>

$\large\quad\text{Define a loss function for learning the conditional likelihood of the data given the observed class}$<br>

In [ ]:
# Define the negative log likelihood cost function
def nll(x_train, distribution):
    return -tf.reduce_mean(distribution.log_prob(x_train))

$\large\quad\text{Define a function to compute the loss and apply the gradients to the trainable parameters}$<br>

In [ ]:
# Define a function to compute the loss and gradients
@tf.function
def get_loss_and_grads(x_train, distribution):
    with tf.GradientTape() as tape:
        tape.watch(distribution.trainable_variables)
        loss = nll(x_train, distribution)
        grads = tape.gradient(loss, distribution.trainable_variables)
    return loss, grads

$\large\quad\text{Generate and learn the conditional likelihood of the data given the observed class }P(X=x | C=c_{k})$<br>

In [ ]:
# Learn the distribution using gradient tape
def make_distribution_withGT(data, labels, nb_classes):

    # Create some empty lists to store the class data, trainable variables, and distributions
    train_vars = []
    distributions = []
    class_data = []
    # Loop over the classes
    for c in range(nb_classes):
        # Instantiate and append randomized trainable variable(s) for distribution of class c 
        train_vars.append(tf.Variable(initial_value=np.random.uniform(low=0.01, high =0.1, size=data.shape[-1])))
        # Instantiate and append a Bernoulli distribution with the trainable variable(s) for class c
        distributions.append(tfp.distributions.Bernoulli(probs=train_vars[c]))
        # Create a mask for the data points belonging to class c
        class_mask = (labels == c)
        # Append the data points belonging to class c to class_data
        class_data.append(data[class_mask, :])
    
    # For each class
    for c_num in range(0,nb_classes):
        # Print the class number
        print('\n%-------------------%')
        print('Class ', c_num)
        print('%-------------------%')

        # Instantiate an optimizer
        optimizer = tf.keras.optimizers.Adam()
        # Use the optimizer to minimize the negative log likelihood for class c
        for i in range(0,100):
            # Compute the loss and gradients of the current distribution
            loss, grads = get_loss_and_grads(class_data[c_num], distributions[c_num])
            # If the iteration is a multiple of 10, print the loss
            if (i%10 == 0):
                print('iter: {}    loss: {}'.format(i, loss))
            # Apply the gradients to the trainable variables of the distribution
            optimizer.apply_gradients(zip(grads, distributions[c_num].trainable_variables))
            # Set eta for clipping minimum
            eta = 1e-3
            # Clip the trainable variables to ensure they are between eta and 1
            clipped_probs = tf.clip_by_value(distributions[c_num].trainable_variables, clip_value_min=eta, clip_value_max=1.)
            # Remove any dimensions of size 1
            train_vars[c_num] = tf.squeeze(clipped_probs)
    # Create nb_classes of independent Bernoulli distributions with the trainable "probs" parameter for each class    
    # Batch shape is (nb_classes,) and event shape is 1
    dist = tfp.distributions.Bernoulli(probs=train_vars)
    # Roll the independent Bernoulli distributions into a single multivariate Bernoulli distribution
    # Batch shape is now 1 and event shape is (nb_classes,))
    dist = tfp.distributions.Independent(dist,reinterpreted_batch_ndims=1)
    # Print the distribution
    print(dist)

    return dist

In [ ]:
# Now train the distributions with gradient tape
GT_dist = make_distribution_withGT(data=train_data, labels=train_labels, nb_classes=len(categories))

<H1>3) Compute classification metrics</H1>

In [ ]:
# Predict log probability of class given the distribution, a test sample, and the class priors
def predict_sample(dist, sample, priors):
    # 1) Compute the class conditional probabilities of the sample
    # P(X=x | C=c_k)
    cond_probs = dist.log_prob(sample)
    # 2) Compute the joint likelihood
    # P(C=c_k, X=x) = P(C=c_k) * P(X=x | C=c_k) = log(P(C=c_k)) + log(P(X=x | C=c_k))
    joint_likelihood = np.(tf.math.log(priors), cond_probs)
    # 3) Compute a normalization factor
    norm_factor = tf.math.reduce_logsumexp(joint_likelihood, axis=-1, keepdims=True)
    # 4) Normalise the joint likelihood and returns the log prob
    log_prob = joint_likelihood - norm_factor

    return log_prob

In [ ]:
# Compare the two results
for dist in [GT_dist]:
    training_probabilities = []
    for sample, label in zip(train_data, train_labels):
        training_probabilities.append(predict_sample(dist, sample, priors))
    training_probabilities = np.asarray(training_probabilities)
    training_predicted_classes = np.argmax(training_probabilities, axis =-1)
    print('Training set f1: ', f1_score(train_labels, predicted_classes, average='macro'))
    testing_probabilities = []
    for sample, label in zip(test_data, test_labels):
        testing_probabilities.append(predict_sample(dist, sample, priors))
    testing_probabilities = np.asarray(testing_probabilities)
    testing_predicted_classes = np.argmax(testing_probabilities, axis =-1)
    print('Test set f1: ', f1_score(test_labels, testing_predicted_classes, average='macro'))